# RVC — Hướng Dẫn Training Giọng Nói Từng Bước (Google Colab)

**Notebook này dành cho người mới**, giải thích rõ từng tham số, chia nhỏ từng bước, tối ưu cho Google Colab với Google Drive để **checkpoint không bị mất khi session reset**.

## Tổng quan pipeline

```
[File .wav của bạn trên Drive]
       ↓
  BƯỚC 0 (Colab): Mount Drive + Clone repo + Cài thư viện + Symlinks
       ↓
  BƯỚC 1: Chuẩn bị dataset (đặt .wav lên Drive)
       ↓
  BƯỚC 2: Khởi tạo Python
       ↓
  BƯỚC 3: Đặt tham số
       ↓
  BƯỚC 4: Preprocess — cắt, chuẩn hóa audio
       ↓
  BƯỚC 5+6: Trích F0 + Hubert features
       ↓
  BƯỚC 7: Huấn luyện G + D (lâu nhất — checkpoint lưu tự động lên Drive)
       ↓
  BƯỚC 8: Tạo FAISS Index
       ↓
  BƯỚC 9: Xuất file .pth nhỏ cho inference
       ↓
[G_*.pth + added_*.index + model_infer.pth — tất cả trên Drive]
```

## Lợi ích của Google Drive
- **Checkpoint tự động lưu lên Drive** — Colab reset không mất dữ liệu
- **Weights (HuBERT, RMVPE, pretrained) chỉ tải 1 lần** — session sau dùng lại
- **Dataset .wav lưu trên Drive** — không cần upload lại mỗi lần

## Resume sau khi Colab reset
Chạy lại **Bước 0.1 → 0.4**, rồi tiếp tục từ **Bước 2** — training tự động resume từ checkpoint cuối.

---
# BƯỚC 0 (Colab): Chuẩn Bị Môi Trường

**Chạy toàn bộ phần này trước khi làm gì khác, kể cả sau khi session bị reset.**

> Trước khi chạy: `Runtime → Change runtime type → GPU (T4 hoặc A100)`

### 0.1 — Mount Google Drive & Kiểm tra GPU

**Tại sao cần Drive?**
Colab session bị reset sau ~12 giờ. Bằng cách mount Drive, toàn bộ checkpoint, weights và dataset
được lưu vĩnh viễn tại `MyDrive/rvc_training/` và không bị mất khi reset.

> Notebook này dùng folder **`rvc_training/`** trên Drive — tách biệt hoàn toàn với `rvc_project/`
> (dùng bởi `RVC_pipeline_colab.ipynb`).

**Tại sao cần GPU?**
Trích HuBERT features, training và inference đều yêu cầu CUDA. CPU chạy được nhưng chậm hơn ~50x.

In [1]:
from google.colab import drive
import torch

drive.mount('/content/drive')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print(f"GPU: {props.name}  |  VRAM: {vram_gb:.1f} GB  |  CUDA {torch.version.cuda}")
    print("\n✅ GPU sẵn sàng. Training sẽ chạy trên CUDA.")
else:
    print("\n⚠️  Không tìm thấy GPU CUDA.")
    print("   Đổi runtime: Runtime → Change runtime type → GPU (T4 hoặc A100)")

import os
DRIVE_BASE = '/content/drive/MyDrive/rvc_training'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"\nDrive base: {DRIVE_BASE}")

Mounted at /content/drive
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4  |  VRAM: 14.6 GB  |  CUDA 12.4

✅ GPU sẵn sàng. Training sẽ chạy trên CUDA.

Drive base: /content/drive/MyDrive/rvc_training


### 0.2 — Clone Repo từ GitHub

Tải mã nguồn RVC (`infer/`, `configs/`, `training_pipeline/`, ...) về `/content/compare_rvc`.

- Nếu thư mục đã tồn tại (session cũ chưa reset hẳn) → chỉ `git pull` để cập nhật.
- Nếu chưa có → `git clone` từ đầu (thường 10–30 giây).

**Tại sao không lưu code lên Drive?** Code clone nhanh (~vài chục MB). Những thứ quan trọng (checkpoint, weights, data) sẽ được symlink sang Drive ở Bước 0.4.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/thanhNgan13/compare_rvc.git'
REPO_DIR = '/content/compare_rvc'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo đã có — cập nhật...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print(f'Clone {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'\nWorking dir: {os.getcwd()}')
print('\nCấu trúc thư mục:')
for item in sorted(os.listdir('.')):
    print(f"  {item}{'/' if os.path.isdir(item) else ''}")

### 0.3 — Cài Đặt Thư Viện

Cài toàn bộ dependencies từ `requirements-py311.txt`. **Phải chạy lại mỗi session mới.**

**Lưu ý:**
- Dùng `requirements-py311.txt` — phiên bản tương thích Python 3.11, dùng fork fairseq ổn định hơn.
- `onnxruntime-gpu` được dùng thay `onnxruntime` để tận dụng GPU trên Colab.
- `ffmpeg` và `aria2` đã có sẵn trên Colab — không cần cài lại.

⏳ **Thời gian:** ~5–10 phút lần đầu.

In [ ]:
import subprocess
from pathlib import Path


def run(cmd):
    print(f'  $ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('FAILED:')
        print((r.stdout + r.stderr)[-3000:])
        raise subprocess.CalledProcessError(r.returncode, cmd)


req_path = Path('/content/compare_rvc/requirements-py311.txt')
if not req_path.exists():
    raise FileNotFoundError('requirements-py311.txt không tìm thấy. Chạy lại Bước 0.2.')

# requirements-py311.txt đã có platform marker tự động:
#   onnxruntime;     sys_platform == 'darwin'  → macOS
#   onnxruntime-gpu; sys_platform != 'darwin'  → Linux/Windows (Colab dùng cái này)
# Không cần lọc tay — pip tự chọn đúng.
print('Cài packages từ requirements-py311.txt...')
run(f'pip install -r {req_path}')

print('\n✅ Cài đặt hoàn tất.')

### 0.4 — Tạo Symlinks: Checkpoint & Weights → Google Drive

**Cơ chế:** Ánh xạ các thư mục quan trọng trong repo sang Drive bằng symbolic link.
Training pipeline viết checkpoint vào `logs/` như bình thường → checkpoint tự động lên Drive.

```
/content/compare_rvc/logs/                → MyDrive/rvc_training/logs/
/content/compare_rvc/assets/weights/      → MyDrive/rvc_training/model_weights/
/content/compare_rvc/assets/hubert/       → MyDrive/rvc_training/assets_cache/hubert/
/content/compare_rvc/assets/rmvpe/        → MyDrive/rvc_training/assets_cache/rmvpe/
/content/compare_rvc/assets/pretrained_v2/→ MyDrive/rvc_training/assets_cache/pretrained_v2/
```

**Tất cả nằm trong `rvc_training/`** — không lẫn vào `rvc_project/` của pipeline notebook.

**Lợi ích:** Weights (HuBERT, RMVPE, pretrained) chỉ cần tải 1 lần. Session sau tạo lại symlink → dùng ngay.

In [ ]:
import os, shutil
from pathlib import Path

REPO  = Path('/content/compare_rvc')
DRIVE = Path(DRIVE_BASE)

SYMLINKS = {
    'logs'                 : DRIVE / 'logs',
    'assets/weights'       : DRIVE / 'model_weights',
    'assets/hubert'        : DRIVE / 'assets_cache/hubert',
    'assets/rmvpe'         : DRIVE / 'assets_cache/rmvpe',
    'assets/pretrained_v2' : DRIVE / 'assets_cache/pretrained_v2',
}

for rel, drive_path in SYMLINKS.items():
    drive_path.mkdir(parents=True, exist_ok=True)
    local = REPO / rel
    local.parent.mkdir(parents=True, exist_ok=True)

    if local.is_symlink():
        local.unlink()
    elif local.is_dir():
        for item in local.iterdir():
            dst = drive_path / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(str(local))

    local.symlink_to(drive_path)
    print(f'  {str(local):<55} → {drive_path}')

print('\n✅ Symlinks OK — checkpoint và weights tự lưu vào Drive.')

### 0.5 — Tải Model Weights (Hubert, RMVPE, Pretrained G/D)

Chỉ chạy nếu lần đầu hoặc weights chưa có trên Drive.
Sau lần đầu, symlink đã trỏ về Drive nên weights được dùng lại tự động.

In [ ]:
import pathlib

CWD = pathlib.Path('/content/compare_rvc').resolve()

weights_can_co = [
    ("assets/hubert/hubert_base.pt",       "Hubert — trích đặc trưng giọng (Bước 6)"),
    ("assets/rmvpe/rmvpe.pt",              "RMVPE — phân tích F0 (Bước 5)"),
    ("assets/pretrained_v2/f0G40k.pth",   "Pretrained Generator 40k (Bước 7)"),
    ("assets/pretrained_v2/f0D40k.pth",   "Pretrained Discriminator 40k (Bước 7)"),
]

print("Kiểm tra model weights:\n")
thieu = []
for duong_dan, mo_ta in weights_can_co:
    f = CWD / duong_dan
    if f.exists():
        size_mb = f.stat().st_size / 1024**2
        print(f"  ✓  {duong_dan} ({size_mb:.0f} MB)")
    else:
        print(f"  ✗  {duong_dan} — THIẾU → sẽ tải")
        thieu.append(duong_dan)

if thieu:
    print(f"\nTải {len(thieu)} file còn thiếu...")
    !python tools/download_assets.py
else:
    print("\n✅ Tất cả weights đã có — không cần tải lại.")

# Kiểm tra logs/mute
mute_dir = CWD / "logs" / "mute"
print(f"\nlogs/mute/: {'✓ OK' if mute_dir.exists() else '✗ THIẾU — sẽ báo lỗi ở Bước 7'}")

---
# BƯỚC 1: Chuẩn Bị Dataset (File .wav của bạn)

**Đây là giọng mà model sẽ học để bắt chước.** Sau khi train xong, khi bạn infer một bài hát, giọng trong bài sẽ được chuyển sang giống giọng này.

**Với Colab, bạn có 2 cách đặt dataset:**
1. ✅ **Dùng Google Drive (khuyến nghị):** Upload .wav lên Drive → đặt đường dẫn Drive vào ô 1.2
2. Upload tạm thời: Dùng `files.upload()` nhưng **sẽ mất khi session reset**

### 1.1 — Yêu cầu về chất lượng audio

| Tiêu chí | Yêu cầu tối thiểu | Khuyến nghị |
|----------|-------------------|-------------|
| Định dạng | `.wav` | `.wav` 16-bit hoặc 32-bit float |
| Sample rate | Bất kỳ (sẽ được resample) | 44.1kHz hoặc 48kHz |
| Nội dung | Giọng nói/hát rõ ràng | Hát, ít nhạc nền |
| Thời lượng tổng | ≥ 5 phút | 10–30 phút |
| Nền ồn | Càng ít càng tốt | Phòng yên tĩnh |
| Số file | ≥ 1 | Nhiều file ngắn (~3-10 phút/file) |

**Lưu ý:**
- Nếu audio có nhạc nền, hãy dùng vocal separator trước (ví dụ: UVR, Demucs)
- Không cần cắt tay — bước Preprocess sẽ tự cắt đoạn im lặng
- **Đặt file .wav vào Google Drive trước**, ví dụ: `MyDrive/rvc_training/datasets/ten_dataset/`

### 1.2 — Chỉ định đường dẫn dataset trên Drive

**Cách 1 (Khuyến nghị — Drive):**
Upload .wav lên `MyDrive/rvc_training/datasets/<ten_dataset>/` rồi chạy ô bên dưới.

**Cách 2 (Tạm thời):**
Đặt `USE_DRIVE_DATASET = False` để upload trực tiếp vào Colab (mất khi reset).

In [ ]:
import pathlib, shutil

# ============================================================
# SỬA ĐÂY
# ============================================================
TEN_DATASET       = "giong_cua_toi"   # Tên thư mục dataset
USE_DRIVE_DATASET = True              # True = dùng Drive, False = upload tạm

# Nếu dùng Drive: đường dẫn thư mục chứa .wav trên Drive
# Mặc định: MyDrive/rvc_training/datasets/<TEN_DATASET>/
# Có thể trỏ đến bất kỳ thư mục nào khác trên Drive
DRIVE_DATASET_PATH = f"/content/drive/MyDrive/rvc_training/datasets/{TEN_DATASET}"
# ============================================================

REPO        = pathlib.Path('/content/compare_rvc')
dataset_dir = REPO / 'datasets' / TEN_DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)

if USE_DRIVE_DATASET:
    src = pathlib.Path(DRIVE_DATASET_PATH)
    if not src.is_dir():
        print(f"⚠️  Chưa có thư mục Drive: {src}")
        print(f"   Tạo thư mục trên Drive và upload .wav vào đó, rồi chạy lại ô này.")
        print(f"   Hoặc đổi DRIVE_DATASET_PATH thành đường dẫn đúng.")
    else:
        if dataset_dir.is_symlink():
            dataset_dir.unlink()
        elif dataset_dir.is_dir() and not any(dataset_dir.iterdir()):
            dataset_dir.rmdir()

        if not dataset_dir.exists():
            dataset_dir.symlink_to(src)
            print(f"✅ Symlink: datasets/{TEN_DATASET}/ → {src}")
        else:
            print(f"✅ Thư mục dataset đã tồn tại: {dataset_dir}")
else:
    from google.colab import files
    print("Upload file .wav của bạn (có thể chọn nhiều file):")
    uploaded = files.upload()
    for filename, data in uploaded.items():
        out_path = dataset_dir / filename
        out_path.write_bytes(data)
        print(f"  Đã lưu: {out_path.name}")

print(f"\nThư mục dataset: {dataset_dir.resolve()}")

### 1.3 — Kiểm tra file .wav và thông tin audio

In [ ]:
import pathlib, warnings
warnings.filterwarnings("ignore")

dataset_dir = pathlib.Path('/content/compare_rvc') / 'datasets' / TEN_DATASET
wavs = sorted(list(dataset_dir.glob("*.wav")) + list(dataset_dir.glob("*.WAV")))

if not wavs:
    print("❌ Không tìm thấy file .wav trong:", dataset_dir.resolve())
    print("   → Kiểm tra lại đường dẫn Drive hoặc upload lại file.")
else:
    print(f"✅ Tìm thấy {len(wavs)} file .wav:\n")
    tong_size = 0
    for f in wavs:
        size_mb = f.stat().st_size / 1024**2
        tong_size += size_mb
        print(f"  {f.name:45s}  {size_mb:6.1f} MB")
    print(f"\n  Tổng: {tong_size:.1f} MB")

    try:
        import librosa
        tong_giay = 0
        print(f"\n{'File':<45} {'Sample Rate':>12} {'Thời lượng':>12}")
        print("-" * 75)
        for f in wavs:
            try:
                y, sr = librosa.load(str(f), sr=None, mono=False)
                tl = (len(y) if y.ndim == 1 else y.shape[1]) / sr
                tong_giay += tl
                print(f"  {f.name:<43} {sr:>12,} Hz {int(tl//60):>4}:{tl%60:05.2f} min")
            except Exception as e:
                print(f"  {f.name:<43} Lỗi đọc: {e}")
        print("-" * 75)
        print(f"Tổng thời lượng: {int(tong_giay//60)} phút {tong_giay%60:.0f} giây")
        if tong_giay < 300:
            print("\n⚠️  Dataset ngắn (< 5 phút). Model có thể kém chất lượng.")
        elif tong_giay < 600:
            print("\n⚠️  Dataset tạm ổn (5-10 phút). Thêm audio nếu có thể.")
        else:
            print("\n✅ Thời lượng dataset tốt.")
    except ImportError:
        print("(librosa chưa sẵn sàng — bỏ qua thống kê thời lượng)")

---
# BƯỚC 2: Khởi Tạo Môi Trường Python

**Chạy một lần** sau khi mở notebook hoặc sau khi Restart kernel / session reset.

Bước này:
1. Đặt working directory về `/content/compare_rvc`
2. Load module `training_pipeline` vào `sys.path`
3. Tạo `config` — chứa thông tin về GPU, Python interpreter, FP16

> **Sau khi session reset:** Luôn chạy lại Bước 0.1 → 0.4, sau đó chạy ô này trước khi tiếp tục.

In [ ]:
import logging
import os
import pathlib
import sys

STANDALONE_ROOT = pathlib.Path('/content/compare_rvc').resolve()
os.chdir(STANDALONE_ROOT)

if not (STANDALONE_ROOT / "infer" / "modules" / "train" / "train.py").is_file():
    raise SystemExit(
        "\n❌ Không tìm thấy mã nguồn RVC!"
        "\n   Chạy lại Bước 0.2 (Clone repo) trước."
    )

if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()
assert root == STANDALONE_ROOT

print("=" * 50)
print("Thư mục gốc  :", STANDALONE_ROOT)
print("Python       :", config.python_cmd)
print("Device Hubert:", config.device)      # cuda:0 hoặc cpu
print("Half-prec    :", config.is_half)      # FP16 (True nếu có GPU hỗ trợ)
print("=" * 50)
print("✅ Bước 2 xong — sẵn sàng cấu hình tham số")

---
# BƯỚC 3: Cấu Hình Tham Số Training

Đây là phần quan trọng nhất. Bạn cần đặt đúng các tham số trước khi train.

### 3.1 — Giải thích chi tiết từng tham số

#### Tham số nhận diện thí nghiệm

| Tham số | Giải thích | Ví dụ |
|---------|-----------|-------|
| `experiment_name` | Tên thư mục chứa output trong `logs/` (trên Drive) | `"giong_A"` |
| `trainset_dir` | Thư mục chứa file .wav gốc | `"datasets/giong_cua_toi"` |

---

#### Tham số về chất lượng âm thanh

| Tham số | Giải thích | Nên chọn gì? |
|---------|-----------|----------------|
| `sample_rate_label` | Sample rate train: `"32k"`, `"40k"`, hoặc `"48k"` | **`"40k"`** cho hầu hết trường hợp |
| `version` | `"v1"` (256 chiều) hoặc `"v2"` (768 chiều) | **`"v2"`** — luôn dùng v2 |

---

#### Tham số về cao độ (F0 — pitch)

| Tham số | Giải thích | Nên chọn gì? |
|---------|-----------|----------------|
| `if_f0` | `True` = train có nhánh cao độ (giữ nốt nhạc khi infer) | **`True`** cho hát |
| `f0_method` | Thuật toán phân tích cao độ | **`"rmvpe"`** — chính xác nhất |

**So sánh f0_method:**

| Phương pháp | Tốc độ | Chính xác | Ghi chú |
|-------------|--------|-----------|--------|
| `rmvpe` | ★★★★ | ★★★★★ | Tốt nhất, cần `rmvpe.pt` |
| `harvest` | ★★ | ★★★★ | Chậm nhưng ổn định |
| `pm` | ★★★★★ | ★★★ | Nhanh nhất, kém chính xác hơn |

---

#### Tham số tài nguyên

| Tham số | Giải thích | Colab T4 |
|---------|-----------|----------|
| `num_processes` | Số CPU core cho preprocess + F0 | `2` (Colab thường có 2 core) |
| `gpu_devices_train` | ID GPU dùng để train | `"0"` |
| `batch_size` | Số mẫu mỗi lần | T4 (15GB): `8–12`, A100: `16–32` |

---

#### Tham số huấn luyện

| Tham số | Giải thích | Gợi ý |
|---------|-----------|-------|
| `total_epochs` | Tổng số vòng lặp | 100–300 cho dataset nhỏ |
| `save_every_epoch` | Lưu checkpoint mỗi N epoch | `10` |

**Gợi ý `batch_size` theo VRAM:**

| VRAM | Batch size |
|------|------------|
| ≤ 4 GB (RTX 3050) | 2–4 |
| ~15 GB (T4) | 8–12 |
| ~40 GB (A100) | 16–32 |

### 3.2 — Đặt tham số (SỬA Ô NÀY)

> **Chỉ cần sửa phần có comment `# SỬA`.**

In [ ]:
from pathlib import Path
from training_pipeline.params import TrainingParams

# ================================================================
# PHẦN CẦN SỬA
# ================================================================

p = TrainingParams(

    # --- Tên thí nghiệm và đường dẫn dataset ---
    experiment_name = "giong_A",                    # SỬA: output lưu vào logs/giong_A/ trên Drive
    trainset_dir    = "datasets/giong_cua_toi",     # SỬA: thư mục chứa .wav (từ Bước 1)

    # --- Sample rate và phiên bản model ---
    sample_rate_label = "40k",   # "32k" | "40k" | "48k" — khuyến nghị "40k"
    version           = "v2",    # "v1" | "v2" — luôn dùng "v2"

    # --- Cao độ F0 ---
    if_f0     = True,       # True = train model giữ cao độ (cần cho hát)
    f0_method = "rmvpe",    # "rmvpe" (tốt nhất) | "harvest" | "pm"

    # --- Tài nguyên Colab ---
    num_processes    = 2,    # CPU core — Colab thường có 2
    gpu_devices_train = "0", # GPU train — Colab chỉ có 1 GPU

    # --- Tham số huấn luyện ---
    total_epochs     = 100,  # SỬA: 50 để test nhanh, 200-300 để chất lượng tốt
    save_every_epoch = 10,   # Lưu checkpoint mỗi N epoch (lên Drive)
    batch_size       = 8,    # SỬA: T4=8, A100=16-32

    # --- FAISS Index ---
    skip_index = False,      # False = tạo index (khuyến nghị)

    # --- Xuất model nhỏ ---
    infer_weight_name = "giong_A_infer",  # Tên file .pth xuất ra (lưu trên Drive)
    extract_infer_pth = False,

    # --- Speaker ID ---
    speaker_id = 0,  # 0 khi train 1 giọng
)

print("✅ Tham số đã đặt xong.")
print("   Tiếp theo: chạy ô 3.3 để xem tóm tắt và kiểm tra.")

### 3.3 — Xem tóm tắt tham số và kiểm tra dataset

In [ ]:
from pathlib import Path
from training_pipeline import steps as train_steps

print("=" * 55)
print("TÓM TẮT THAM SỐ TRAINING")
print("=" * 55)
print(f"  Thí nghiệm   : {p.experiment_name}")
print(f"  Output (Drive): rvc_training/logs/{p.experiment_name}/")
print(f"  Dataset dir  : {p.trainset_dir}")
print(f"  Sample rate  : {p.sample_rate_label}")
print(f"  Version      : {p.version}")
print(f"  Có F0 (pitch): {p.if_f0}")
print(f"  F0 method    : {p.f0_method}")
print(f"  CPU processes: {p.num_processes}")
print(f"  GPU train    : {p.gpu_devices_train}")
print(f"  Epochs       : {p.total_epochs}")
print(f"  Save mỗi     : {p.save_every_epoch} epoch")
print(f"  Batch size   : {p.batch_size}")
print(f"  FAISS index  : {'BỎ QUA' if p.skip_index else 'SẼ TẠO'}")
print(f"  Model infer  : rvc_training/model_weights/{p.infer_weight_name}.pth")
print("=" * 55)

# Kiểm tra dataset
ts = STANDALONE_ROOT / p.trainset_dir
if not ts.is_dir():
    print(f"\n❌ Thư mục dataset không tồn tại: {ts}")
    print("   Kiểm tra lại Bước 1.2")
else:
    wavs = list(ts.glob("*.wav")) + list(ts.glob("*.WAV"))
    if not wavs:
        print(f"\n❌ Không có file .wav trong {ts}")
    else:
        print(f"\n✅ Dataset: {len(wavs)} file .wav")

# Kiểm tra mute templates
mm = train_steps.check_mute_template(STANDALONE_ROOT)
if mm:
    print(f"\n❌ logs/mute/ thiếu: {mm}")
else:
    print("✅ logs/mute/: OK")

print("\n→ Nếu tất cả ✅, chạy BƯỚC 4 (Preprocess).")

---
# BƯỚC 4: Preprocess — Cắt và Chuẩn Hóa Audio

**Mục đích:** Đọc file .wav gốc → cắt đoạn im lặng → chia thành clip ngắn (~3-5 giây)
→ chuẩn hóa âm lượng → lưu 2 bản:
- `logs/<tên>/0_gt_wavs/` — audio ở sample rate bạn chọn → **Ground Truth dùng để train**
- `logs/<tên>/1_16k_wavs/` — audio ở 16kHz → **Dùng cho Hubert ở Bước 6**

**Lưu ý Colab:** Kết quả lưu trong `logs/` trên Drive → không cần chạy lại sau khi reset session.

**Thời gian:** Thường 1-5 phút tùy dung lượng dataset.

In [ ]:
exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name
gt_wavs_existing = list((exp_dir / "0_gt_wavs").glob("*.wav")) if (exp_dir / "0_gt_wavs").exists() else []

if gt_wavs_existing:
    print(f"Preprocess: đã có {len(gt_wavs_existing)} clip trên Drive — bỏ qua.")
    print("  (Xóa thư mục logs/<tên>/0_gt_wavs/ nếu muốn chạy lại)")
else:
    print("Bắt đầu Preprocess...")
    print(f"  Input : {p.trainset_dir}")
    print(f"  Output: logs/{p.experiment_name}/0_gt_wavs/ và 1_16k_wavs/")
    print()
    train_steps.step_preprocess(STANDALONE_ROOT, config, p)
    print("\n✅ Bước 4 (Preprocess) hoàn thành.")

### 4.1 — Kiểm tra kết quả Preprocess

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

gt_wavs   = list((exp_dir / "0_gt_wavs").glob("*.wav"))  if (exp_dir / "0_gt_wavs").exists()  else []
_16k_wavs = list((exp_dir / "1_16k_wavs").glob("*.wav")) if (exp_dir / "1_16k_wavs").exists() else []

print("Kết quả Preprocess:")
print(f"  0_gt_wavs/   : {len(gt_wavs)} file")
print(f"  1_16k_wavs/  : {len(_16k_wavs)} file")

if gt_wavs:
    print(f"\nVí dụ 5 clip đầu:")
    for f in sorted(gt_wavs)[:5]:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.0f} KB)")
    if len(gt_wavs) > 5:
        print(f"  ... và {len(gt_wavs)-5} file khác")

    if len(gt_wavs) < 10:
        print("\n⚠️  Ít clip. Dataset có thể quá ngắn hoặc phần lớn là im lặng.")
    else:
        print(f"\n✅ Preprocess tạo ra {len(gt_wavs)} clip — sẵn sàng cho Bước 5+6.")
else:
    print("\n❌ Không có file trong 0_gt_wavs/. Xem log:")
    log_file = exp_dir / "preprocess.log"
    if log_file.exists():
        print(log_file.read_text(encoding='utf-8', errors='ignore')[-2000:])

---
# BƯỚC 5+6: Trích Xuất F0 và Đặc Trưng Hubert

Hai bước này được thực hiện cùng nhau trong một lần gọi.

**Bước 5 — F0 là gì?** Fundamental Frequency — cao độ cơ bản của giọng. Nếu model biết F0,
khi infer nó có thể **giữ đúng nốt nhạc** của bài gốc. Output:
- `logs/<tên>/2a_f0/` — F0 ở dạng Hz (`.npy`)
- `logs/<tên>/2b-f0nsf/` — F0 normalized dùng cho NSF vocoder

**Bước 6 — Hubert là gì?** Model của Meta học "nội dung" giọng nói (phoneme, âm tiết) không phụ thuộc vào giọng ai. RVC dùng Hubert để encode giọng nguồn → decode ra giọng đích.
- Tải `assets/hubert/hubert_base.pt` lên GPU
- Encode từng clip thành vector 768 chiều → lưu vào `logs/<tên>/3_feature768/`

**Lưu ý Colab:** Kết quả lưu trên Drive → **không cần chạy lại** sau reset session.

**Thời gian:** Khoảng 15–30 phút với 10 phút audio trên T4.

In [ ]:
exp_dir          = STANDALONE_ROOT / "logs" / p.experiment_name
feature_dir_name = "3_feature768" if p.version == "v2" else "3_feature256"
feature_dir      = exp_dir / feature_dir_name
features_existing = list(feature_dir.glob("*.npy")) if feature_dir.exists() else []

if features_existing:
    print(f"Features: đã có {len(features_existing)} file trên Drive — bỏ qua.")
    print("  (Xóa thư mục 3_feature768/ nếu muốn chạy lại)")
else:
    print(f"Bắt đầu trích F0 ({p.f0_method}) + Hubert features...")
    print(f"  Hubert   : assets/hubert/hubert_base.pt")
    print(f"  Output   : logs/{p.experiment_name}/2a_f0/, 2b-f0nsf/, {feature_dir_name}/")
    print()
    train_steps.step_extract_f0_and_features(STANDALONE_ROOT, config, p)
    print("\n✅ Bước 5+6 (F0 + Hubert features) hoàn thành.")

### 6.1 — Kiểm tra kết quả

In [ ]:
from pathlib import Path

exp_dir          = STANDALONE_ROOT / "logs" / p.experiment_name
feature_dir_name = "3_feature768" if p.version == "v2" else "3_feature256"

f0_files      = list((exp_dir / "2a_f0").glob("*.npy"))       if (exp_dir / "2a_f0").exists()       else []
f0nsf_files   = list((exp_dir / "2b-f0nsf").glob("*.npy"))    if (exp_dir / "2b-f0nsf").exists()    else []
feature_files = list((exp_dir / feature_dir_name).glob("*.npy")) if (exp_dir / feature_dir_name).exists() else []
gt_wavs       = list((exp_dir / "0_gt_wavs").glob("*.wav"))   if (exp_dir / "0_gt_wavs").exists()   else []

print("Kết quả Bước 5+6:")
print(f"  2a_f0/          : {len(f0_files)} file .npy"     + (" (bỏ qua vì if_f0=False)" if not p.if_f0 else ""))
print(f"  2b-f0nsf/       : {len(f0nsf_files)} file .npy"  + (" (bỏ qua vì if_f0=False)" if not p.if_f0 else ""))
print(f"  {feature_dir_name}/: {len(feature_files)} file .npy")
print(f"  gt_wavs         : {len(gt_wavs)} file .wav")

if len(feature_files) == len(gt_wavs) > 0:
    print(f"\n✅ {len(feature_files)} feature files khớp — sẵn sàng train (Bước 7).")
elif len(feature_files) != len(gt_wavs):
    print(f"\n⚠️  Số feature ({len(feature_files)}) khác số gt_wav ({len(gt_wavs)}).")
    print("   Một số file có thể bị lỗi. Xem extract_f0_feature.log")
else:
    print("\n❌ Không có feature files. Chạy lại Bước 5+6.")

---
# BƯỚC 7: Huấn Luyện Model (Train)

**Đây là bước lâu nhất.** Train 2 mạng:
- **Generator (G)**: học cách chuyển (Hubert feature + F0 + speaker) → waveform giống giọng đích
- **Discriminator (D)**: học cách phân biệt âm thanh thật/giả → ép G phải giỏi hơn

**Checkpoint tự động lưu lên Drive** mỗi `save_every_epoch` epoch.
Nếu Colab reset, chạy lại Bước 0.1→0.4 + Bước 2, rồi chạy lại ô này — training tự resume từ checkpoint cuối.

**Ước tính thời gian trên T4:**
- 100 epoch với 10 phút audio, batch_size=8: ~2–4 giờ

**Theo dõi tiến trình:**
Mở file `Drive/rvc_project/logs/<tên>/train.log` để xem log realtime.

### 7.1 — Kiểm tra dữ liệu trước khi train

In [ ]:
from pathlib import Path

exp_dir          = STANDALONE_ROOT / "logs" / p.experiment_name
feature_dir_name = "3_feature768" if p.version == "v2" else "3_feature256"

cac_kiem_tra = [
    ("0_gt_wavs",      "*.wav",  "Ground truth wavs (Bước 4)"),
    ("1_16k_wavs",     "*.wav",  "16kHz wavs (Bước 4)"),
    (feature_dir_name, "*.npy",  "Hubert features (Bước 6)"),
]
if p.if_f0:
    cac_kiem_tra += [
        ("2a_f0",    "*.npy",  "F0 raw (Bước 5)"),
        ("2b-f0nsf", "*.npy",  "F0 NSF (Bước 5)"),
    ]

print("Kiểm tra dữ liệu trước khi train:\n")
tong_ok = True
for ten_thu_muc, pattern, mo_ta in cac_kiem_tra:
    d     = exp_dir / ten_thu_muc
    files = list(d.glob(pattern)) if d.exists() else []
    ok    = len(files) > 0
    icon  = "✓" if ok else "✗"
    print(f"  {icon}  {ten_thu_muc:18s} {len(files):4d} file  ({mo_ta})")
    if not ok:
        tong_ok = False

print()
if tong_ok:
    print("✅ Tất cả dữ liệu sẵn sàng. Có thể train.")
else:
    print("❌ Thiếu dữ liệu. Chạy lại bước bị thiếu trước khi train.")

### 7.2 — Chạy Training

> **⚠️ Ô này chạy rất lâu (nhiều giờ).** Kernel sẽ bị block trong suốt thời gian train.
>
> Checkpoint được lưu tự động lên Drive mỗi `save_every_epoch` epoch.
> Nếu session reset, training sẽ tự resume từ checkpoint cuối khi bạn chạy lại ô này.

In [ ]:
print("Bắt đầu Training...")
print(f"  Experiment : {p.experiment_name}")
print(f"  Epochs     : {p.total_epochs}")
print(f"  Batch size : {p.batch_size}")
print(f"  GPU        : {p.gpu_devices_train}")
print(f"  Checkpoint : mỗi {p.save_every_epoch} epoch → rvc_training/logs/{p.experiment_name}/")
print(f"  Log file   : rvc_training/logs/{p.experiment_name}/train.log")
print()
print("⏳ Đang train... (kernel sẽ block cho đến khi xong hoặc session timeout)")
print("   Nếu session bị reset: chạy lại 0.1→0.4 + Bước 2 + ô này để resume tự động.")

train_steps.step_train(STANDALONE_ROOT, config, p)

print("\n✅ Bước 7 (Training) hoàn thành.")

### 7.3 — Xem log train và checkpoint đã lưu

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

train_log = exp_dir / "train.log"
if train_log.exists():
    lines = train_log.read_text(encoding='utf-8', errors='ignore').splitlines()
    print(f"=== train.log (40 dòng cuối / tổng {len(lines)} dòng) ===")
    for line in lines[-40:]:
        print(line)
else:
    print("train.log chưa có.")

print()
g_checkpoints = sorted(exp_dir.glob("G_*.pth"))
if g_checkpoints:
    print(f"Checkpoints G đã lưu ({len(g_checkpoints)} file):")
    for f in g_checkpoints:
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name}  ({size_mb:.0f} MB)")
else:
    print("Chưa có checkpoint G nào.")

---
# BƯỚC 8: Tạo FAISS Index

**FAISS Index dùng để làm gì?** Khi infer, RVC có thể dùng **retrieval** — tìm vector feature
gần nhất trong tập training để blend vào. Điều này giúp giọng infer **tự nhiên hơn**.

**Bước này làm gì:**
- Gom tất cả vector từ `3_feature768/`
- Train FAISS index (IVF + Flat)
- Lưu `logs/<tên>/added_IVF...index` → **file này dùng khi infer**

**Lưu ý Colab:** Index lưu trên Drive → **không cần chạy lại** sau reset session.

**Thời gian:** Thường 1-5 phút.

In [ ]:
exp_dir          = STANDALONE_ROOT / "logs" / p.experiment_name
existing_indexes = list(exp_dir.glob("added_*.index"))

if existing_indexes:
    print("FAISS index đã có trên Drive:")
    for f in existing_indexes:
        print(f"  {f.name}  ({f.stat().st_size/1024**2:.1f} MB)")
elif p.skip_index:
    print("skip_index=True — Bỏ qua Bước 8.")
    print("Khi infer, đặt index_rate=0 để không dùng retrieval.")
else:
    print("Bắt đầu tạo FAISS index...")
    print(f"  Input : logs/{p.experiment_name}/3_feature768/")
    print(f"  Output: logs/{p.experiment_name}/added_IVF...index (trên Drive)")
    print()

    for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
        print(line)

    print("\n✅ Bước 8 (FAISS Index) hoàn thành.")

### 8.1 — Kiểm tra file index

In [ ]:
from pathlib import Path

if p.skip_index:
    print("Đã bỏ qua bước index.")
else:
    exp_dir         = STANDALONE_ROOT / "logs" / p.experiment_name
    added_indexes   = list(exp_dir.glob("added_*.index"))
    trained_indexes = list(exp_dir.glob("trained_*.index"))

    print("File index:")
    for f in trained_indexes:
        print(f"  (trung gian) {f.name}  ({f.stat().st_size/1024**2:.1f} MB)")
    for f in added_indexes:
        print(f"  ✅ (DÙNG ĐỂ INFER) {f.name}  ({f.stat().st_size/1024**2:.1f} MB)")

    if not added_indexes:
        print("❌ Không tìm thấy added_*.index.")
    else:
        print(f"\n✅ Index sẵn sàng trên Drive.")

---
# BƯỚC 9: Xuất Model Nhỏ cho Inference

**Tại sao cần xuất?** File `G_*.pth` từ bước train rất nặng (~200-600MB) và chứa cả optimizer state.
Để infer, ta chỉ cần **trọng số mạng G** → tạo file `.pth` nhỏ hơn (~60-70MB) trong `assets/weights/` (trên Drive).

**File output:** `MyDrive/rvc_training/model_weights/<infer_weight_name>.pth`

### 9.1 — Chọn checkpoint để xuất

In [ ]:
from pathlib import Path

exp_dir       = STANDALONE_ROOT / "logs" / p.experiment_name
g_checkpoints = sorted(exp_dir.glob("G_*.pth"))

if not g_checkpoints:
    print("❌ Chưa có checkpoint G nào. Chạy Bước 7 trước.")
else:
    print(f"Các checkpoint G trong logs/{p.experiment_name}/ (trên Drive):\n")
    for i, f in enumerate(g_checkpoints):
        size_mb = f.stat().st_size / 1024**2
        tag     = " ← MỚI NHẤT" if i == len(g_checkpoints) - 1 else ""
        print(f"  [{i}] {f.name}  ({size_mb:.0f} MB){tag}")
    print()
    print("→ Mặc định sẽ xuất checkpoint MỚI NHẤT.")
    print("  Nếu muốn xuất checkpoint cụ thể, đặt:")
    print(f'     p.g_checkpoint_for_extract = "{g_checkpoints[0].name}"')

### 9.2 — Xuất model

In [ ]:
# Đặt checkpoint muốn xuất (để trống = tự chọn mới nhất)
p.g_checkpoint_for_extract = ""   # Hoặc đặt tên file cụ thể, ví dụ: "G_12500.pth"

ck_str = 'mới nhất' if not p.g_checkpoint_for_extract else p.g_checkpoint_for_extract
print(f"Xuất model nhỏ từ checkpoint {ck_str}")
print(f"Output: rvc_training/model_weights/{p.infer_weight_name}.pth")
print()

result = train_steps.step_extract_small_weights(STANDALONE_ROOT, p)
print(result)

output_file = STANDALONE_ROOT / "assets" / "weights" / f"{p.infer_weight_name}.pth"
if output_file.exists():
    size_mb = output_file.stat().st_size / 1024**2
    print(f"\n✅ Model infer đã xuất: {output_file.name}  ({size_mb:.1f} MB)")
    print(f"   Lưu trên Drive: MyDrive/rvc_training/model_weights/{p.infer_weight_name}.pth")
else:
    print("\n❌ Không tìm thấy file output. Kiểm tra lỗi ở trên.")

---
# TỔNG KẾT — Các File Output Sau Khi Train Xong

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

print("=" * 65)
print("TỔNG KẾT CÁC FILE OUTPUT (trên Google Drive)")
print("=" * 65)

# 1. Model weights (cho inference)
weights_dir = STANDALONE_ROOT / "assets" / "weights"
weights     = list(weights_dir.glob("*.pth")) if weights_dir.exists() else []
print("\n[1] Model cho inference (MyDrive/rvc_training/model_weights/):")
for f in weights:
    size_mb = f.stat().st_size / 1024**2
    print(f"    ✅ {f.name}  ({size_mb:.1f} MB)")
if not weights:
    print("    (chưa có — chạy Bước 9)")

# 2. Checkpoints
g_ckpts = sorted(exp_dir.glob("G_*.pth"))
print(f"\n[2] Checkpoints G (MyDrive/rvc_training/logs/{p.experiment_name}/):")
for f in g_ckpts:
    size_mb = f.stat().st_size / 1024**2
    print(f"    {f.name}  ({size_mb:.0f} MB)")
if not g_ckpts:
    print("    (chưa có — chạy Bước 7)")

# 3. FAISS Index
indexes = list(exp_dir.glob("added_*.index"))
print(f"\n[3] FAISS Index (MyDrive/rvc_training/logs/{p.experiment_name}/):")
for f in indexes:
    size_mb = f.stat().st_size / 1024**2
    print(f"    ✅ {f.name}  ({size_mb:.1f} MB)")
if not indexes:
    print("    (chưa có — chạy Bước 8)")

print("\n" + "=" * 65)
print("Để INFER (chuyển giọng), bạn cần:")
print(f"  • Model  : MyDrive/rvc_training/model_weights/{p.infer_weight_name}.pth")
if indexes:
    print(f"  • Index  : {indexes[-1].name}")
print("  • Notebook: RVC_infer_notebook.ipynb hoặc RVC_pipeline_colab.ipynb")
print("=" * 65)

print("\n📁 Cấu trúc thư mục trên Drive sau khi train:")
print("   MyDrive/rvc_training/                    ← folder riêng, không lẫn rvc_project/")
print(f"   ├── logs/{p.experiment_name}/")
print( "   │   ├── G_*.pth, D_*.pth               ← Checkpoint đầy đủ")
print( "   │   ├── added_*.index                  ← FAISS index cho inference")
print( "   │   └── 0_gt_wavs/, 3_feature768/, ... ← Features đã trích")
print( "   ├── model_weights/                      ← model_infer.pth nhỏ (~70MB)")
print( "   ├── assets_cache/hubert/                ← HuBERT weights (dùng lại)")
print( "   ├── assets_cache/rmvpe/                 ← RMVPE weights (dùng lại)")
print( "   ├── assets_cache/pretrained_v2/         ← Pretrained G/D (dùng lại)")
print( "   └── datasets/<ten_dataset>/             ← File .wav gốc của bạn")

---
## Hướng Dẫn Resume Sau Khi Colab Reset

Vì checkpoint và data đã trên Drive (`rvc_training/`), việc resume chỉ cần:

```
Bước 0.1 — Mount Drive + GPU check
Bước 0.2 — Clone repo (git pull nếu đã có)
Bước 0.3 — Cài thư viện (bắt buộc mỗi session)
Bước 0.4 — Tạo lại symlink về Drive (rvc_training/)
Bước 2   — Khởi tạo Python
Bước 3.2 — Đặt tham số (giống lần trước)
Bước 7.2 — Chạy Training (tự resume từ checkpoint cuối)
```

Bước 4, 5+6, 8 **không cần chạy lại** — kết quả đã có trên Drive.

---

## So Sánh Folder Drive với Notebook Khác

| Folder Drive | Dùng bởi | Nội dung |
|---|---|---|
| `MyDrive/rvc_training/` | **Notebook này** | Dataset, checkpoint, weights train từng bước |
| `MyDrive/rvc_project/` | `RVC_pipeline_colab.ipynb` | Pipeline đầu cuối + đánh giá |

Hai folder hoàn toàn độc lập — không lẫn dữ liệu vào nhau.

---

## Xử Lý Sự Cố Thường Gặp

| Triệu chứng | Nguyên nhân có thể | Cách khắc phục |
|------------|-------------------|----------------|
| `RuntimeError: extract_feature_print.py thất bại` | fairseq không tương thích | Kiểm tra phiên bản fairseq/hydra, xem log chi tiết |
| OOM (out of memory) | Batch size quá lớn | Giảm `batch_size` xuống 4–6 với T4 |
| Training rất chậm | Chạy trên CPU | Đổi runtime → GPU |
| Không tìm thấy weights | Symlink chưa tạo hoặc Drive chưa mount | Chạy lại Bước 0.1 + 0.4 |
| Dataset .wav không tìm thấy | Đường dẫn Drive sai | Kiểm tra lại `DRIVE_DATASET_PATH` ở Bước 1.2 |
| Checkpoint không tăng epoch | Đang resume đúng | Bình thường — training tiếp tục từ epoch cũ |